# v0.8.0 -- Transactions (core)

`SurrealDBConnectionManager.transaction()` groups several writes into **one atomic
`BEGIN TRANSACTION; ... COMMIT TRANSACTION;`** block. Pass `tx=` to `save()`, `update()`,
`merge()` and `delete()` to enrol them; if the `async with` block raises, nothing is written.

This notebook uses an **HTTP** connection, which always selects the *buffered* strategy that
v0.8.0 shipped:

- operations are buffered and flushed as a single query at commit,
- a `save()` inside the transaction needs an **explicit record id** (auto-ids land in v0.9.0),
- **reads inside the transaction are not possible** and raise a clear error.

The v0.9.0 notebook shows the native *interactive* strategy you get over WebSocket on
SurrealDB 3.x, where reads and auto-ids work.

## 1. Connect (HTTP -> buffered strategy)

In [1]:
import os
from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# HTTP forces the BufferedTransaction strategy on every server version -- this is exactly the
# v0.8.0 model. (Over WebSocket on SurrealDB 3.x, transaction() would auto-upgrade to the
# native interactive strategy shown in the v0.9.0 notebook.)
SurrealDBConnectionManager.set_connection(
    url=f"http://{HOST}:{PORT}",
    user="root", password="root",
    namespace="examples", database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Define a model and reset the demo table

Setup is idempotent: a `DELETE` on a table that was never created raises `NotFoundError` on SurrealDB 3.x, so we suppress it.

In [2]:
import contextlib

from pydantic import Field

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Account(BaseSurrealModel):
    model_config = SurrealConfigDict(primary_key="id")
    id: str | None = None
    owner: str
    balance: int = 0


client = await SurrealDBConnectionManager.get_client()
with contextlib.suppress(Exception):
    await client.query("DELETE Account;", {})
print("table reset")

table reset


## 3. Commit several writes atomically

Both `save()` calls are buffered and committed together when the block exits.

In [3]:
async with SurrealDBConnectionManager.transaction() as tx:
    await Account(id="alice", owner="Alice", balance=100).save(tx=tx)
    await Account(id="bob", owner="Bob", balance=0).save(tx=tx)

rows = await client.query("SELECT owner, balance FROM Account ORDER BY owner;", {})
for r in rows:
    print(r["owner"], "->", r["balance"])

Alice -> 100
Bob -> 0


## 4. An atomic transfer with `merge(tx=)`

Move 30 from Alice to Bob. Either both balances change or neither does. In a buffered transaction `merge()` also applies the new values to the in-memory instance immediately.

In [4]:
alice = Account(id="alice", owner="Alice", balance=100)
bob = Account(id="bob", owner="Bob", balance=0)

async with SurrealDBConnectionManager.transaction() as tx:
    await alice.merge(tx=tx, balance=70)
    await bob.merge(tx=tx, balance=30)

print("in-memory after commit:", alice.balance, bob.balance)
rows = await client.query("SELECT owner, balance FROM Account ORDER BY owner;", {})
print("in DB:", {r["owner"]: r["balance"] for r in rows})

in-memory after commit: 70 30
in DB: {'Alice': 70, 'Bob': 30}


## 5. A Python exception rolls everything back

If the block raises before committing, the buffered writes are discarded -- the debit below never reaches the database.

In [5]:
try:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Account(id="alice", owner="Alice", balance=0).update(tx=tx)  # would zero Alice
        raise RuntimeError("aborting the transfer")
except RuntimeError as exc:
    print("rolled back:", exc)

rows = await client.query("SELECT owner, balance FROM Account WHERE id = Account:alice;", {})
print("Alice balance unchanged:", rows[0]["balance"])

rolled back: aborting the transfer
Alice balance unchanged: 70


## 6. A server-side failure rolls back the whole batch

Two `CREATE`s with the same id can't both succeed; the conflict aborts the transaction and the ORM raises `SurrealDbError`.

In [6]:
from surreal_orm_lite import SurrealDbError

try:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Account(id="carol", owner="Carol", balance=5).save(tx=tx)
        await Account(id="carol", owner="Carol again", balance=9).save(tx=tx)
except SurrealDbError as exc:
    print("transaction failed:", exc)

rows = await client.query("SELECT * FROM Account WHERE id = Account:carol;", {})
print("Carol persisted:", len(rows))  # 0 -> the whole batch rolled back

transaction failed: Transaction failed and rolled back: Database record `Account:carol` already exists
Carol persisted: 0


## 7. Buffered-strategy caveats

Two things are intentionally unavailable in a buffered transaction. Both raise a clear error rather than failing silently.

In [7]:
# (a) save() needs an explicit id (auto-ids arrive with the interactive strategy in v0.9.0)
try:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Account(owner="No Id", balance=1).save(tx=tx)
except SurrealDbError as exc:
    print("no-id save:", exc)

# (b) reads inside a buffered transaction are not supported
try:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Account.objects(tx=tx).filter(owner="Alice").exec()
except SurrealDbError as exc:
    print("read in buffered tx:", exc)

no-id save: save(tx=...) requires an explicit id on a buffered transaction (auto-id requires a WebSocket connection to SurrealDB 3.x).
read in buffered tx: Reads inside a transaction require a WebSocket connection to SurrealDB 3.x (native interactive transactions); the current transaction is buffered (HTTP or SurrealDB 2.6.x).


## 8. Cleanup

In [8]:
await client.query("DELETE Account;", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
